# Extending Agents with MCP and Plugin Architectures

The CDA library ships eleven builtin tools — file I/O, shell, grep, glob, memory, and a few more. These cover the standard coding-agent use case well, but real-world agents often need to query databases, call REST APIs, search the web, or interact with cloud services. Adding each integration as a bespoke CDA tool requires writing schema definitions, execution logic, error handling, and registration code from scratch — and the result is non-portable.

The **Model Context Protocol** (MCP) is an open standard that decouples tool *definition* from tool *consumption*. You define tools once on an MCP server; any compliant client can connect and use them automatically. The analogy is USB: a standard connector so you don't need custom wiring for every device.

This notebook covers: why a fixed tool set isn't enough, the MCP architecture, building an MCP server with FastMCP, how the CDA library bridges MCP servers via `MCPManager`, HTTP transport for remote servers, and the agent-as-tool composition pattern.

## The Extensibility Problem

Consider what it takes to add a "query the company database" tool to the CDA library today:

1. Define a Pydantic `BaseModel` for the parameters
2. Subclass `Tool` and implement `execute()`
3. Choose a `ToolKind` and decide on approval semantics
4. Register the tool in the `ToolRegistry`
5. Repeat for every agent that needs it

This is fine for one or two tools, but it becomes a maintenance problem when you have dozens. It's also non-portable: the tool definition is coupled to the CDA library's internal API, so it can't be shared with agents built in LangChain, AutoGen, or any other framework.

MCP separates these concerns:
- The *server* owns the tool definition: schema, description, execution logic
- The *client* discovers and calls tools through a standard protocol
- The server is framework-agnostic — any compliant client can connect

:::{.callout-note}
MCP was developed by Anthropic and released as an open standard in late 2024. It is now supported by most major agent frameworks, including LangChain, AutoGen, and the OpenAI Agents SDK.

:::

## The MCP Protocol

The MCP protocol defines a client-server interaction with three phases:

**Connection.** The client connects to the server over a transport. The server reports its name and capabilities.

**Tool discovery.** The client calls `list_tools()`. The server returns a list of tool objects, each with a `name`, `description`, and `inputSchema` (JSON Schema). The client registers these in its own tool system.

**Tool invocation.** When the LLM requests a tool call, the client calls `call_tool(name, args)`. The server executes the tool and returns a result as a list of content items (text, images, etc.).

The protocol is transport-agnostic:

| Transport | Use case | Mechanism |
|-----------|----------|-----------|
| **stdio** | Local tools | Client spawns server subprocess; talks via stdin/stdout |
| **HTTP (Streamable HTTP)** | Remote services | Client connects to URL; HTTP POST + SSE |

The same client code works with either transport — only the configuration changes.

## Setup

**Setup.** Imports for the MCP bridge and agent components:

In [ ]:
import os
import asyncio
import json
import textwrap
from pathlib import Path
from dotenv import load_dotenv

from fastmcp import FastMCP, Client
from fastmcp.client import PythonStdioTransport

from notebooks.agent.config import Config, ApprovalPolicy
from notebooks.agent.session import Session
from notebooks.agent.agent import Agent
from notebooks.agent.events import AgentEventType
from notebooks.agent.tools.registry import create_default_registry
from notebooks.agent.tools.mcp_bridge import MCPManager, MCPServerConfig

load_dotenv()

We reuse the agent runner helper from previous notebooks:

In [ ]:
async def run_agent(task: str, config: Config) -> tuple[str, dict]:
    """Run a CDA Agent on *task* and return (response, usage)."""
    session = Session(config)
    agent = Agent(config=config, session=session)
    response = ""
    usage = {}
    async for event in agent.run(task):
        if event.type == AgentEventType.AGENT_END:
            response = event.data.get("response", "") or ""
            usage = event.data.get("usage") or {}
    return response, usage

## Building an MCP Server

Before connecting an agent to an MCP server, we build one from scratch. This demystifies the server side and shows how little boilerplate MCP requires with FastMCP.

We write a `math_server.py` script to the `tmp/` directory. The server exposes three tools: arithmetic operations that will be called by the agent:

In [ ]:
MATH_SERVER_PATH = Path("tmp/math_server.py")  # (1)
MATH_SERVER_PATH.parent.mkdir(parents=True, exist_ok=True)

MATH_SERVER_PATH.write_text(textwrap.dedent("""
    from fastmcp import FastMCP

    server = FastMCP("Math Server")

    @server.tool()
    def add(a: float, b: float) -> float:
        \"\"\"Add two numbers.\"\"\"
        return a + b

    @server.tool()
    def multiply(a: float, b: float) -> float:
        \"\"\"Multiply two numbers.\"\"\"
        return a * b

    @server.tool()
    def power(base: float, exponent: float) -> float:
        \"\"\"Raise base to the power of exponent.\"\"\"
        return base ** exponent

    if __name__ == "__main__":
        server.run()
""").strip())

print(f"Wrote server to {MATH_SERVER_PATH}")

1. We write the server script to `tmp/` rather than the project root to keep it out of version control. It is a temporary helper for this notebook.

The `@server.tool()` decorator is all that's needed to register a function as an MCP tool. FastMCP infers the JSON Schema from the function signature and docstring. The server can be launched as `python math_server.py` for standalone use, or — as we'll see — spawned by the MCP client as a subprocess.

### Direct client interaction

Before wiring the server into a CDA agent, we connect a raw FastMCP client to verify tool discovery and invocation:

In [ ]:
transport = PythonStdioTransport(
    script_path=str(MATH_SERVER_PATH),
    args=[],
)

async with Client(transport) as client:
    tools = await client.list_tools()  # (1)
    print("Discovered tools:")
    for tool in tools:
        print(f"  {tool.name}: {tool.description}")
    print()

    result = await client.call_tool("add", {"a": 37, "b": 43})  # (2)
    print(f"add(37, 43) = {result.content[0].text}")

    result = await client.call_tool("power", {"base": 2, "exponent": 10})
    print(f"power(2, 10) = {result.content[0].text}")

1. `list_tools()` returns `mcp.types.Tool` objects with `name`, `description`, and `inputSchema`.
2. `call_tool()` returns a `CallToolResult` whose `content` is a list of content items — for text tools, each item has a `.text` attribute.

### Adding a stateful tool

To show that MCP tools can do more than pure arithmetic, we extend the server with a tool that reads from a "database" — a JSON file. We write a second server script:

In [ ]:
DB_PATH = Path("tmp/users.json")
DB_PATH.write_text(json.dumps({
    "1": {"name": "Alice", "role": "engineer", "team": "infra"},
    "2": {"name": "Bob", "role": "manager", "team": "platform"},
    "3": {"name": "Carol", "role": "engineer", "team": "ml"},
}))

DB_SERVER_PATH = Path("tmp/db_server.py")
DB_SERVER_PATH.write_text(textwrap.dedent(f"""
    import json
    from pathlib import Path
    from fastmcp import FastMCP

    server = FastMCP("DB Server")
    DB_PATH = Path("{DB_PATH.resolve()}")

    @server.tool()
    def get_user(user_id: str) -> dict:
        \"\"\"Retrieve a user record by ID. Returns name, role, and team.\"\"\"
        data = json.loads(DB_PATH.read_text())
        user = data.get(str(user_id))
        if user is None:
            return {{"error": f"User {{user_id}} not found"}}
        return user

    @server.tool()
    def list_users() -> list:
        \"\"\"List all users with their IDs, names, and roles.\"\"\"
        data = json.loads(DB_PATH.read_text())
        return [
            {{"id": uid, **record}}
            for uid, record in data.items()
        ]

    if __name__ == "__main__":
        server.run()
""").strip())

print(f"Wrote DB server to {DB_SERVER_PATH}")

**Verify.** Connect and call the DB tools:

In [ ]:
db_transport = PythonStdioTransport(script_path=str(DB_SERVER_PATH), args=[])

async with Client(db_transport) as db_client:
    tools = await db_client.list_tools()
    print("DB tools:", [t.name for t in tools])

    result = await db_client.call_tool("get_user", {"user_id": "1"})
    print("User 1:", result.content[0].text)

    result = await db_client.call_tool("list_users", {})
    print("All users:", result.content[0].text)

## The MCP Bridge

The CDA library provides three classes in `tools/mcp_bridge.py` that bridge MCP servers into the tool registry:

**`MCPServerConfig`** holds connection parameters for one server. For stdio, set `command` and `args`; for HTTP, set `url`. The two transports are mutually exclusive.

**`MCPToolAdapter`** wraps a single MCP tool as a CDA `Tool` subclass. It:
- Names the tool as `{server_name}__{tool_name}` to avoid collisions across servers
- Sets `kind = ToolKind.NETWORK` so approval logic treats it like a network call
- Prefixes the description with `[server_name]` for clear attribution
- Stores the raw JSON Schema from the server's `inputSchema` as its schema
- Delegates `execute()` to `client.call_tool()`

**`MCPManager`** owns the connection lifecycle. Call `connect_all(registry)` to discover and register tools, and `close()` on shutdown. It accepts a `skip_errors=True` flag so a single unavailable server doesn't abort startup.

### Connecting the math server

We create a `MCPManager` with our math server, connect it to the default tool registry, and verify the tools are registered:

In [ ]:
math_cfg = MCPServerConfig(
    name="math",
    command="uv",
    args=["run", "python", str(MATH_SERVER_PATH.resolve())],  # (1)
)

config = Config(approval=ApprovalPolicy.YOLO, max_turns=5)
registry = create_default_registry(config)

manager = MCPManager([math_cfg])
registered = await manager.connect_all(registry)

print("Registered tools by server:")
for server, tools in registered.items():
    print(f"  {server}: {tools}")

print("\nManager status:")
print(json.dumps(manager.status(), indent=2))

1. We use `uv run python` to ensure the server runs in the project's virtual environment, which has `fastmcp` available.

The three math tools are now registered as `math__add`, `math__multiply`, and `math__power`. They are indistinguishable from builtin tools in the registry — both are `Tool` instances with a `name`, `description`, and `execute()` method.

**Agent task.** We now run a CDA agent with this registry and give it a task that requires calling the MCP tools:

In [ ]:
session = Session(config, registry=registry)  # (1)
agent = Agent(config=config, session=session)

task = "What is 37 + 43? Then compute 6 raised to the power of 4."
response = ""
async for event in agent.run(task):
    if event.type == AgentEventType.AGENT_END:
        response = event.data.get("response", "") or ""

print(response)

1. We pass the pre-built `registry` to `Session` so it picks up the MCP tools rather than creating a new default registry without them.

The agent calls `math__add` and `math__power` — MCP tools — without knowing or caring whether they are builtin or external. From the agent's perspective, they are just names in the tool list.

**Cleanup.** Always close the MCPManager when finished to release the subprocess connections:

In [ ]:
await manager.close()
print("MCP connections closed.")

## Agent-as-Tool Pattern

MCP enables a powerful composition pattern: wrapping an entire agent as an MCP tool. Any MCP client — including agents built with different frameworks — can then invoke the agent as a black-box tool.

This is complementary to NB09's `SubAgentTool`. That pattern is internal: the parent agent spawns children through the CDA library's own abstractions. The agent-as-tool pattern is external: the inner agent is exposed through the MCP protocol, so the outer caller could be on a different machine, in a different language, or using a different framework.

We build an MCP server that exposes a single tool `ask_agent(question)`. Internally it spins up a CDA agent and returns its response:

In [ ]:
AGENT_SERVER_PATH = Path("tmp/agent_server.py")
AGENT_SERVER_PATH.write_text(textwrap.dedent("""
    import asyncio
    from fastmcp import FastMCP
    from notebooks.agent.config import Config, ApprovalPolicy
    from notebooks.agent.session import Session
    from notebooks.agent.agent import Agent
    from notebooks.agent.events import AgentEventType

    server = FastMCP("Agent Server")

    @server.tool()
    async def ask_agent(question: str) -> str:
        \"\"\"Ask a CDA coding agent a question and return its response.\"\"\"
        config = Config(approval=ApprovalPolicy.YOLO, max_turns=10)
        session = Session(config)
        agent = Agent(config=config, session=session)
        response = ""
        async for event in agent.run(question):
            if event.type == AgentEventType.AGENT_END:
                response = event.data.get("response", "") or ""
        return response or "(no response)"

    if __name__ == "__main__":
        server.run()
""").strip())

print(f"Wrote agent server to {AGENT_SERVER_PATH}")

**Invoke directly via FastMCP client.** We connect a raw FastMCP client to verify the agent server responds correctly before wiring it into a CDA agent:

In [ ]:
agent_transport = PythonStdioTransport(
    script_path=str(AGENT_SERVER_PATH.resolve()),
    args=[],
)

async with Client(agent_transport) as agent_client:
    tools = await agent_client.list_tools()
    print("Agent server tools:", [t.name for t in tools])
    print()

    result = await agent_client.call_tool(
        "ask_agent",
        {"question": "How many Python files are in src/notebooks/agent/?"}
    )
    print("Agent response:")
    print(result.content[0].text)

The inner CDA agent runs to completion, calls its file tools, and returns its findings. All of this happens inside an MCP tool call — the outer client does not need to know anything about the CDA library.

:::{.callout-note}
The agent-as-tool pattern makes the inner agent opaque to the outer caller. If the inner agent fails, the outer system sees only a tool error. For production systems, ensure the inner agent logs its reasoning independently and has clear error propagation.

:::

### When to use agent-as-tool

| Use it when | Avoid it when |
|-------------|---------------|
| Agents need to be shared across teams or codebases | The caller and callee share the same codebase |
| The inner agent runs on different infrastructure | Latency matters — MCP adds protocol overhead |
| You want framework-agnostic composition | The inner agent's reasoning needs to be audited by the caller |
| You're building an agent marketplace or registry | NB09's SubAgentTool already meets the need |

## Putting It Together

We conclude with a full end-to-end demonstration: a CDA agent whose tool registry contains both the standard builtin tools and tools from two MCP servers — the math server and the DB server. The agent uses both seamlessly on a single task.

**Registry assembly.** We create a fresh config and registry, then attach both MCP servers:

In [ ]:
combined_config = Config(approval=ApprovalPolicy.YOLO, max_turns=10)
combined_registry = create_default_registry(combined_config)

combined_manager = MCPManager([
    MCPServerConfig(
        name="math",
        command="uv",
        args=["run", "python", str(MATH_SERVER_PATH.resolve())],
    ),
    MCPServerConfig(
        name="db",
        command="uv",
        args=["run", "python", str(DB_SERVER_PATH.resolve())],
    ),
])

registered = await combined_manager.connect_all(combined_registry)
for server, tools in registered.items():
    print(f"{server}: {tools}")

**Hybrid task.** We give the agent a task that requires both the DB server (to look up a user) and the math server (to compute a value from the result):

In [ ]:
combined_session = Session(combined_config, registry=combined_registry)
combined_agent = Agent(config=combined_config, session=combined_session)

hybrid_task = (
    "First, list all users from the database. "
    "Then compute the sum of all user IDs using the math tools. "
    "Finally, raise that sum to the power of 2 and report all three results."
)

hybrid_response = ""
async for event in combined_agent.run(hybrid_task):
    if event.type == AgentEventType.AGENT_END:
        hybrid_response = event.data.get("response", "") or ""

print(hybrid_response)

The agent's tool list contains `read_file`, `write_file`, `shell`, and the other CDA builtins alongside `math__add`, `math__multiply`, `math__power`, `db__get_user`, and `db__list_users`. It picks the right tool for each step without any explicit routing logic — the names and descriptions are sufficient for the LLM to make the right calls.

**Cleanup.** Close the MCP connections:

In [ ]:
await combined_manager.close()
print("All MCP connections closed.")

## Summary

The CDA library's tool system is open-ended by design. The `MCPManager` bridge means that any MCP server — whether a local subprocess or a remote HTTP service — contributes tools to the registry with no library modifications required:

| Approach | When to use |
|----------|-------------|
| Builtin tools | Standard file, shell, and memory operations |
| Custom `Tool` subclass | Tight integration with CDA internals; project-specific tools |
| MCP stdio server | Local tools written in Python (or any language) |
| MCP HTTP server | Remote services, shared tools, cross-team APIs |
| Agent-as-tool | Framework-agnostic agent composition across infrastructure boundaries |

:::{.callout-note}
The MCP ecosystem is growing rapidly. Public MCP servers exist for web search, GitHub, Slack, databases, and many other services. The `MCPManager` can connect to any of them with a one-line config change.

:::

---

■